> **How to run.** Open in Google Colab, authenticate to Earth Engine in cell 1, then
> **Runtime → Run all**. The notebook detects that no clean post-fire scene exists yet and
> skips burn severity automatically — it will not error. Outputs are stripped from this file;
> running it regenerates `facts_la_mierla_2026.json`, the FIRMS timeline, and the map.
> Expected result: 1,525 detections over 7 days, ignition 1.2 km from La Mierla village,
> Sentinel-1 cross-check negative.

# La Mierla 2026 — Sierra Norte de Guadalajara

**A fire that is still burning.** This is the Tenerife template with one difference: it
detects that no clean post-fire Sentinel-2 scene exists yet and **skips burn severity
automatically** instead of crashing. Run All today and you get a publishable piece.

| | |
|---|---|
| Detected | **16 July 2026, 13:55 local** (fixed lookout) |
| Size | ~32,000 ha as of 23 July — Spain's largest fire of 2026 |
| Status | stabilising, still active |
| Protected | reached the **core zone** of the Sierra Norte de Guadalajara Natural Park |
| Fuel | coniferous forest + sclerophyllous scrub |
| Validation | **Copernicus EMS EMSR898** (activated 19 Jul 2026, 22:00) |

### What works today, and what does not

| Layer | Today | Why |
|---|---|---|
| FIRMS hotspots + progression | works | thermal — smoke is irrelevant |
| Sentinel-1 SAR scar | works | C-band penetrates smoke |
| Sentinel-2 burn severity | blocked | needs a clear post-fire scene |
| Validation vs EMS | pending | delineation publishes in days |

When a clear scene lands, set `post_start`, Run All again, and severity appears with no
other change.


## 0 - Setup

In [ ]:
# !pip install -q earthengine-api geemap pandas requests

import ee, geemap, json, io, requests
import pandas as pd
import datetime as dt

try:
    ee.Initialize()
except Exception:
    ee.Authenticate(); ee.Initialize()   # or ee.Initialize(project='your-gcp-project-id')
print('Earth Engine ready:', ee.__version__)

## 1 - CONFIG

`containment_date` stays `None` while the fire burns, so the FIRMS window runs to today
automatically. Set a real date once it is out.

Note the separate **S1 post window**: for an active fire the radar "after" pass is the most
recent one available, not a future date.

In [ ]:
CONFIG = {
    # --- identity ---
    'fire_id':          'la_mierla_2026',
    'fire_name':        'La Mierla, Sierra Norte de Guadalajara',
    'region':           'Castilla-La Mancha, Spain',
    'year':             2026,
    'ignition_date':    '2026-07-16',
    'containment_date': None,              # None = still burning -> FIRMS runs to today

    # --- area of interest [W, S, E, N] ---
    'aoi_bbox':   [-3.45, 40.95, -2.80, 41.40],

    # --- Sentinel-2 windows ---
    'pre_start':  '2026-06-05', 'pre_end':  '2026-07-15',
    'post_start': '2026-08-10', 'post_end': '2026-09-10',   # placeholder; no clear scene yet

    # --- Sentinel-1 windows (post = most recent passes; works during the fire) ---
    's1_pre_start':  '2026-06-05', 's1_pre_end':  '2026-07-15',
    's1_post_start': '2026-07-18', 's1_post_end': None,      # None = today

    # --- official reference ---
    'official_source':  'Copernicus EMS EMSR898',
    'official_area_ha': 0,                 # fill in when the delineation publishes

    # --- FIRMS ---
    'firms_map_key': '38da194a340f838919a9d282c513b640',
    'firms_source':  'VIIRS_SNPP_NRT',     # MUST be NRT - the _SP archive stops 2026-04-27

    # ================= FIXED FOR ALL FIRES =================
    's2_collection':      'COPERNICUS/S2_SR_HARMONIZED',
    'cs_plus_collection': 'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED',
    'cs_band':'cs_cdf', 'cs_threshold':0.60, 'scl_shadow_backup':True,
    'max_cloud_pct':70, 'scale_m':20, 'apply_offset':True,
    's1_pass':'DESCENDING', 's1_pol':'VH', 's1_confirm_db':0.3,
    'sev_thresholds': {'unburned_max':0.15,'low_max':0.27,
                       'moderate_low_max':0.44,'moderate_high_max':0.66},
}

TODAY = dt.date.today().isoformat()
END   = CONFIG['containment_date'] or TODAY          # FIRMS window end

AOI      = ee.Geometry.Rectangle(CONFIG['aoi_bbox'])
AOI_BUFF = AOI.buffer(10000)
T        = CONFIG['sev_thresholds']

print('AOI area (km2):', round(AOI.area(1).getInfo()/1e6, 1))
print('fire window   :', CONFIG['ignition_date'], '->', END,
      '(still active)' if not CONFIG['containment_date'] else '')

## 2 - Sentinel-2 -> NBR, and the decision that drives everything

If the post-fire window returns **zero usable scenes**, `HAS_SEVERITY` becomes `False` and
every severity cell below skips quietly. Nothing crashes; nothing is guessed.

In [ ]:
def mask_s2_clouds(img):
    good = img.select(CONFIG['cs_band']).gte(CONFIG['cs_threshold'])
    if CONFIG['scl_shadow_backup']:
        good = good.And(img.select('SCL').neq(3))
    return img.updateMask(good)

def nbr(img):
    return img.normalizedDifference(['B8','B12']).rename('NBR')

def s2_nbr_composite(start, end):
    csplus = ee.ImageCollection(CONFIG['cs_plus_collection'])
    col = (ee.ImageCollection(CONFIG['s2_collection'])
           .filterBounds(AOI_BUFF).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CONFIG['max_cloud_pct']))
           .linkCollection(csplus, [CONFIG['cs_band']]).map(mask_s2_clouds))
    return col.map(nbr).select('NBR').median().clip(AOI_BUFF), col.size()

nbr_pre,  n_pre  = s2_nbr_composite(CONFIG['pre_start'],  CONFIG['pre_end'])
nbr_post, n_post = s2_nbr_composite(CONFIG['post_start'], CONFIG['post_end'])

N_PRE  = int(n_pre.getInfo())
N_POST = int(n_post.getInfo())
HAS_SEVERITY = N_POST > 0

print(f'Pre scenes: {N_PRE} | Post scenes: {N_POST}')
if HAS_SEVERITY:
    print('-> severity WILL be computed.')
else:
    print('-> no clean post-fire scene yet. Severity skipped automatically.')
    print('   Publishing detections + SAR only. Re-run once a clear scene exists.')

## 3 - dNBR + RBR + severity   *(runs only if imagery exists)*

In [ ]:
severity = burn_mask = dnbr = rbr = None
offset_val = 0.0

if HAS_SEVERITY:
    worldcover_full = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(AOI_BUFF)
    land_mask = worldcover_full.neq(80)

    dnbr_raw = nbr_pre.subtract(nbr_post).rename('dNBR').updateMask(land_mask)

    if CONFIG['apply_offset']:
        control = AOI.buffer(8000).difference(AOI.buffer(3000))
        stats = dnbr_raw.reduceRegion(
            ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
            control, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).getInfo()
        offset_val = stats.get('dNBR_mean') or 0.0
        print('dNBR offset:', round(offset_val,4), '| control pixels:', stats.get('dNBR_count'))

    dnbr = dnbr_raw.subtract(offset_val).rename('dNBR')
    rbr  = dnbr.divide(nbr_pre.add(1.001)).rename('RBR')

    severity = (dnbr
        .where(dnbr.lt(T['unburned_max']), 0)
        .where(dnbr.gte(T['unburned_max']).And(dnbr.lt(T['low_max'])), 1)
        .where(dnbr.gte(T['low_max']).And(dnbr.lt(T['moderate_low_max'])), 2)
        .where(dnbr.gte(T['moderate_low_max']).And(dnbr.lt(T['moderate_high_max'])), 3)
        .where(dnbr.gte(T['moderate_high_max']), 4)
        .toInt().updateMask(land_mask).rename('severity'))
    burn_mask = severity.gte(1)
    print('dNBR / RBR / severity ready.')
else:
    print('skipped - no post-fire imagery.')

## 4 - Severity area + normalized shares

In [ ]:
SEV_NAMES = {1:'low',2:'moderate_low',3:'moderate_high',4:'high'}
severity_ha, severity_pct = {}, {}
total_burned_ha = pct_high = mean_rbr = None

if HAS_SEVERITY:
    grp = (ee.Image.pixelArea().divide(1e4).addBands(severity)
           .reduceRegion(ee.Reducer.sum().group(1,'class'),
                         AOI, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).getInfo())
    severity_ha = {v:0.0 for v in SEV_NAMES.values()}
    for g in grp.get('groups', []):
        c = int(g['class'])
        if c in SEV_NAMES: severity_ha[SEV_NAMES[c]] = round(g['sum'],1)

    total_burned_ha = round(sum(severity_ha.values()), 1)
    severity_pct = {k:(round(100*v/total_burned_ha,1) if total_burned_ha else 0.0)
                    for k,v in severity_ha.items()}
    pct_high = severity_pct['high']
    mean_rbr = round(ee.Number(rbr.updateMask(burn_mask).reduceRegion(
        ee.Reducer.mean(), AOI, CONFIG['scale_m'],
        maxPixels=1e13, bestEffort=True).get('RBR')).getInfo() or 0.0, 4)

    print('Total burned (ha):', total_burned_ha, '| % high:', pct_high, '| mean RBR:', mean_rbr)
else:
    print('skipped.')

## 5 - Land-cover breakdown

In [ ]:
WC_NAMES = {10:'tree_cover',20:'shrubland',30:'grassland',40:'cropland',50:'built_up',
            60:'bare_sparse',70:'snow_ice',80:'water',90:'wetland',95:'mangrove',100:'moss_lichen'}
landcover_burned_ha, landcover_burned_pct = {}, {}

if HAS_SEVERITY:
    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(AOI)
    lcg = (ee.Image.pixelArea().divide(1e4).addBands(worldcover.updateMask(burn_mask))
           .reduceRegion(ee.Reducer.sum().group(1,'lc'),
                         AOI, CONFIG['scale_m'], maxPixels=1e13, bestEffort=True).getInfo())
    for g in lcg.get('groups', []):
        landcover_burned_ha[WC_NAMES.get(int(g['lc']), f"class_{int(g['lc'])}")] = round(g['sum'],1)
    for key in ('tree_cover','shrubland','cropland'):
        landcover_burned_ha.setdefault(key, 0.0)
    landcover_burned_pct = {k:(round(100*v/total_burned_ha,1) if total_burned_ha else 0.0)
                            for k,v in landcover_burned_ha.items()}
    print('Burned by land cover (ha):', landcover_burned_ha)
else:
    print('skipped.')

## 6 - Protected-area impact - exposure, not damage

The Sierra Norte de Guadalajara Natural Park is in the WDPA. The regional government said
the fire reached its **core zone**; this is the measured version of that claim.

In [ ]:
protected_area_burned_ha = protected_share_pct = None
protected_areas = []

wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons').filterBounds(AOI)
protected_areas = wdpa.aggregate_array('NAME').distinct().getInfo()[:10]
print('Protected areas intersecting the AOI:', protected_areas)

if HAS_SEVERITY:
    protected_mask = ee.Image.constant(0).byte().paint(wdpa,1).gt(0).clip(AOI)
    protected_area_burned_ha = round(ee.Number(
        ee.Image.pixelArea().divide(1e4).updateMask(burn_mask).updateMask(protected_mask)
          .reduceRegion(ee.Reducer.sum(), AOI, CONFIG['scale_m'],
                        maxPixels=1e13, bestEffort=True).get('area')).getInfo() or 0.0, 1)
    protected_share_pct = (round(100*protected_area_burned_ha/total_burned_ha,1)
                           if total_burned_ha else 0.0)
    print('Protected ha burned:', protected_area_burned_ha, f'({protected_share_pct}%)')
else:
    print('protected-area hectares: skipped (needs the burn mask).')

## 7 - Validation vs Copernicus EMS EMSR898

In [ ]:
official = CONFIG['official_area_ha']
pct_diff = None

if HAS_SEVERITY and official:
    pct_diff = round(100*(total_burned_ha-official)/official, 1)
    print(f"Ours: {total_burned_ha} ha | {CONFIG['official_source']}: {official} ha "
          f"| diff: {pct_diff:+.1f}%")
elif not official:
    print(f"{CONFIG['official_source']}: activated, delineation not yet in CONFIG.")
    print('Fill official_area_ha when the EMS product publishes.')
else:
    print('no measured area yet - nothing to validate.')

## 8 - Sentinel-1 cross-check: the layer that works through smoke

Optical is blind while the fire burns. C-band radar is not. The "after" pass is the most
recent available, so this runs **today**.

Without an optical burn mask, this thresholds the SAR change itself to give an *indicative*
extent. That is not a validated burned-area figure and must not be presented as one.

In [ ]:
S1_POST_END = CONFIG['s1_post_end'] or TODAY

def s1_mean(start, end):
    col = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filterBounds(AOI).filterDate(start, end)
           .filter(ee.Filter.eq('instrumentMode','IW'))
           .filter(ee.Filter.eq('orbitProperties_pass', CONFIG['s1_pass']))
           .filter(ee.Filter.listContains('transmitterReceiverPolarisation', CONFIG['s1_pol']))
           .select(CONFIG['s1_pol']))
    return col.mean().clip(AOI), col.size()

s1_pre,  s1n_pre  = s1_mean(CONFIG['s1_pre_start'],  CONFIG['s1_pre_end'])
s1_post, s1n_post = s1_mean(CONFIG['s1_post_start'], S1_POST_END)
s1_change = s1_post.subtract(s1_pre).rename('dVH')

n1, n2 = s1n_pre.getInfo(), s1n_post.getInfo()
print('S1 scenes  pre:', n1, '| post:', n2)

sar_confirms_scar = None
sar_delta_db = None
sar_change_area_ha = None

if n1 > 0 and n2 > 0:
    if HAS_SEVERITY:
        def _mean(mask):
            return ee.Number(s1_change.updateMask(mask).reduceRegion(
                ee.Reducer.mean(), AOI, 30, maxPixels=1e13, bestEffort=True).get('dVH')).getInfo()
        cb, cu = _mean(burn_mask) or 0.0, _mean(burn_mask.Not()) or 0.0
        sar_delta_db = round(cb - cu, 3)
    else:
        drop = s1_change.lt(-1.0)          # more than 1 dB VH loss
        sar_change_area_ha = round(ee.Number(
            ee.Image.pixelArea().divide(1e4).updateMask(drop)
              .reduceRegion(ee.Reducer.sum(), AOI, 30,
                            maxPixels=1e13, bestEffort=True).get('area')).getInfo() or 0.0, 1)
        sar_delta_db = round(ee.Number(s1_change.updateMask(drop).reduceRegion(
            ee.Reducer.mean(), AOI, 30, maxPixels=1e13, bestEffort=True)
            .get('dVH')).getInfo() or 0.0, 3)
        print(f'SAR change area (>1 dB VH loss): {sar_change_area_ha:,.0f} ha')
        print('  -> INDICATIVE extent only. Not a validated burned-area figure.')

    if sar_delta_db is not None and abs(sar_delta_db) >= CONFIG['s1_confirm_db']:
        sar_confirms_scar = True
    print(f'VH change delta: {sar_delta_db} dB | confirms: {sar_confirms_scar}')
else:
    print('No S1 scene in one of the windows - sar_confirms_scar = None (inconclusive)')

## 9 - FIRMS: ignition point, spread path, fire power

`VIIRS_SNPP_NRT`, because the `_SP` archive ends 2026-04-27. The window runs to **today**
while the fire is active, so re-run this cell daily.

In [ ]:
w, s, e, n = CONFIG['aoi_bbox']
area = f"{w},{s},{e},{n}"
d0 = dt.datetime.fromisoformat(CONFIG['ignition_date'])
d1 = dt.datetime.fromisoformat(END)

frames, cur = [], d0
while cur <= d1:
    span = min(5, (d1-cur).days + 1)          # FIRMS area API allows 1..5 days per call
    url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
           f"{CONFIG['firms_map_key']}/{CONFIG['firms_source']}/{area}/{span}/{cur.date()}")
    try:
        r = requests.get(url, timeout=60)
        if r.ok and 'latitude' in r.text:
            frames.append(pd.read_csv(io.StringIO(r.text)))
        elif not r.ok:
            print('FIRMS', cur.date(), '->', r.status_code, r.text[:90])
    except Exception as ex:
        print('FIRMS fetch failed for', cur.date(), ':', ex)
    cur += dt.timedelta(days=span)

firms = {'ignition_lat':None,'ignition_lon':None,'first_detection':None,
         'last_detection':None,'active_days':None,'n_detections':0,
         'frp_max_mw':None,'frp_total_mw':None,'max_spread_km':None}

if frames:
    df = pd.concat(frames, ignore_index=True).drop_duplicates()
    df['ts'] = pd.to_datetime(df['acq_date'].astype(str) + ' ' +
                              df['acq_time'].astype(str).str.zfill(4).str[:2] + ':' +
                              df['acq_time'].astype(str).str.zfill(4).str[2:], errors='coerce')
    df = df.dropna(subset=['ts']).sort_values('ts').reset_index(drop=True)

    conf_col   = 'confidence' if 'confidence' in df.columns else None
    frp_col    = 'frp' if 'frp' in df.columns else None
    bright_col = next((c for c in ('bright_ti4','brightness') if c in df.columns), None)
    print('columns ->', 'confidence:', bool(conf_col), '| frp:', bool(frp_col),
          '| brightness:', bright_col)

    ig = df.iloc[0]
    ig_lat, ig_lon = round(float(ig['latitude']),5), round(float(ig['longitude']),5)

    import math
    def km_from_ignition(lo, la):
        R = 6371
        p1, p2 = math.radians(ig_lat), math.radians(la)
        dp, dl = p2-p1, math.radians(lo-ig_lon)
        h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
        return R*2*math.atan2(math.sqrt(h), math.sqrt(1-h))
    dmax = max(km_from_ignition(r.longitude, r.latitude) for r in df.itertuples(index=False))

    firms = {
        'ignition_lat':    ig_lat,
        'ignition_lon':    ig_lon,
        'first_detection': str(df['ts'].min()),
        'last_detection':  str(df['ts'].max()),
        'active_days':     int((df['ts'].max()-df['ts'].min()).days) + 1,
        'n_detections':    int(len(df)),
        'frp_max_mw':      round(float(df[frp_col].max()), 1) if frp_col else None,
        'frp_total_mw':    round(float(df[frp_col].sum()), 1) if frp_col else None,
        'max_spread_km':   round(dmax, 1),
    }

    feats = []
    for r in df.itertuples(index=False):
        props = {'date': str(r.ts),
                 'confidence': str(getattr(r, conf_col)) if conf_col else ''}
        if frp_col:    props['frp']      = float(getattr(r, frp_col) or 0)
        if bright_col: props['bright_k'] = float(getattr(r, bright_col) or 0)
        feats.append({'type':'Feature',
                      'geometry':{'type':'Point',
                                  'coordinates':[float(r.longitude), float(r.latitude)]},
                      'properties':props})
    with open(f"firms_timeline_{CONFIG['fire_id']}.geojson", 'w') as f:
        json.dump({'type':'FeatureCollection','features':feats}, f)

    print('FIRMS:', firms)
    print('ignition point:', ig_lat, ig_lon)
    if frp_col:
        print(f"FRP: max {firms['frp_max_mw']} MW | total {firms['frp_total_mw']} MW")
    print(f"furthest detection from ignition: {dmax:.1f} km")

    pad = 0.01
    if (df['longitude'].min() < w+pad or df['longitude'].max() > e-pad or
        df['latitude'].min()  < s+pad or df['latitude'].max()  > n-pad):
        print('WARNING: detections reach the AOI edge - widen aoi_bbox and re-run.')
else:
    print('No FIRMS data. Check MAP_KEY, and that firms_source is an *_NRT product.')

## 10 - Visual QA

In [ ]:
m = geemap.Map(); m.centerObject(AOI, 10)
if HAS_SEVERITY:
    m.addLayer(severity.updateMask(burn_mask),
               {'min':1,'max':4,'palette':['ffe08a','f9a03f','e8532b','9d0208']}, 'Severity')
m.addLayer(s1_change, {'min':-3,'max':3,'palette':['red','white','blue']}, 'S1 VH change')
if firms['ignition_lat'] is not None:
    m.addLayer(ee.Geometry.Point([firms['ignition_lon'], firms['ignition_lat']]),
               {'color':'black'}, 'Ignition')
m.addLayer(AOI, {'color':'gray'}, 'AOI', False)
m

## 11 - Export facts.json - same locked schema, nulls where nothing was measured

In [ ]:
limits = ['Measured while the fire was still active; figures describe the situation to date.']
if HAS_SEVERITY:
    limits.append('dNBR estimates severity, not a field-surveyed perimeter.')
else:
    limits.append('No clean post-fire Sentinel-2 scene yet, so burn severity is not reported.')
limits += [
    'FIRMS points are ~375 m detections; the ignition point is the first detected pixel, not the exact origin.',
    'Sentinel-1 change is an indicative extent, not a validated burned-area figure.',
    'Protected-area figures are exposure (burn within boundaries), not ecological damage.',
    'Nothing here counts people, homes or ecological loss; none of that was measured.',
]

facts = {
    'fire_id':CONFIG['fire_id'], 'fire_name':CONFIG['fire_name'], 'region':CONFIG['region'],
    'year':CONFIG['year'], 'ignition_date':CONFIG['ignition_date'],
    'containment_date':CONFIG['containment_date'],
    'status':'contained' if CONFIG['containment_date'] else 'active at time of measurement',
    'sensors':'Sentinel-2 (severity), Sentinel-1 (cross-check), VIIRS FIRMS (timeline)',
    'method':'dNBR (USGS/Key-Benson) + RBR (Parks et al. 2014); S1 VH change; FIRMS ignition+path',
    'pre_fire_window':[CONFIG['pre_start'],CONFIG['pre_end']],
    'post_fire_window':[CONFIG['post_start'],CONFIG['post_end']],
    'severity_measured':HAS_SEVERITY,
    'dnbr_offset_applied':round(offset_val,4),

    'total_burned_ha':total_burned_ha,
    'severity_ha':severity_ha, 'severity_pct':severity_pct, 'pct_high_severity':pct_high,
    'mean_rbr':mean_rbr,

    'landcover_burned_ha':landcover_burned_ha, 'landcover_burned_pct':landcover_burned_pct,
    'protected_area_burned_ha':protected_area_burned_ha,
    'protected_share_pct':protected_share_pct, 'protected_areas':protected_areas,

    'official_reference':{'source':CONFIG['official_source'],
                          'area_ha':CONFIG['official_area_ha'] or None,
                          'our_area_ha':total_burned_ha, 'pct_diff':pct_diff},

    'sar_confirms_scar':sar_confirms_scar, 'sar_delta_db':sar_delta_db,
    'sar_change_area_ha':sar_change_area_ha,
    'firms':firms,

    's2_scenes_pre':N_PRE, 's2_scenes_post':N_POST,
    'limitations':limits,
    'generated_utc':dt.datetime.utcnow().isoformat()+'Z',
}
with open(f"facts_{CONFIG['fire_id']}.json",'w') as f:
    json.dump(facts, f, indent=2, ensure_ascii=False)
print(json.dumps(facts, indent=2, ensure_ascii=False))

## 12 - Build the interactive map

`groundtruth_map.build` skips the severity layer automatically when `severity=None`.

In [ ]:
!wget -q -O groundtruth_map.py https://raw.githubusercontent.com/Sanjusajimon220/wildfire-eo-llm/main/groundtruth_map.py

import importlib, groundtruth_map as gm
importlib.reload(gm)

PLACES = [
    ['Tamajon',               41.0308, -3.2278],
    ['La Mierla',             41.0500, -3.1333],
    ['Cogolludo',             41.0242, -3.0906],
    ['Umbralejo',             41.1608, -3.2331],
    ['Almiruete',             41.1200, -3.1900],
    ['Arroyo de las Fraguas', 41.1450, -3.0950],
    ['Zarzuela de Jadraque',  41.1000, -2.9500],
]

gm.build(CONFIG, AOI, PLACES, severity=severity, burn_mask=burn_mask, px=1400)

## What to do next

**Today** - download `fire_spread_la_mierla_2026.html` and `facts_la_mierla_2026.json`,
draft the article with `02_generate_article.py --prompt`, verify it, publish as a live
chapter. The honest framing: *this is what a satellite can tell you while a fire is still
burning, and what it cannot.*

**Daily while it burns** - re-run cells 9 to 12. The FIRMS window extends to today
automatically, so the progression keeps growing.

**When it is out** - set `containment_date`, and when a clear Sentinel-2 scene exists set
`post_start` / `post_end` and Run All. `HAS_SEVERITY` flips to `True`, severity computes,
and the same notebook produces the full piece with no other edits.

**When EMSR898 publishes** - put the hectares in `official_area_ha` and the validation panel
fills in.

### Honest limits of the live version
- Sentinel-1 change is an **indicative extent**, not a validated area. Do not compare it to
  the ~32,000 ha in news reports as though the two measure the same thing.
- FIRMS detections are 375 m and only mark where fire was hot enough to see at overpass time.
- Protected-area hectares need the optical burn mask, so they wait for severity.
